# Do Transcript-Level and Protein-Level Responsiveness Agree?

This is one chapter of the *From Sequence to Variant: Protein Language Models on CFDE Data* training module. Follow the numbered chapters in order.


---
## Addendum: Do Transcript-Level and Protein-Level Responsiveness Agree?

Since this module defines "exercise-responsive" using transcriptomics (Part 2) but models *protein* sequence with ESM2 (Part 3-4), it's worth directly checking: for the same genes, does MoTrPAC's **proteomics** data call the same genes responsive? RNA expression and protein abundance don't always move together (due to translation efficiency, protein degradation, and
other post-transcriptional regulation), so some disagreement is expected, the question is how much.

This is a lightweight check, not a full re-run of the pipeline on proteomics data though the above workflow can be similarly extended for proteomics data.

In [ ]:
# ============================================================
# CELL PURPOSE: compare our transcriptomics-based "exercise-responsive" gene calls against
# MoTrPAC's proteomics data for the same genes, to see how often the two agree.
# ============================================================
# This reuses the same tarball we already downloaded in Part 2 -- PROT_LIVER_DA is structured
# identically to TRNSCRPT_LIVER_DA, just for protein abundance instead of RNA expression.

prot_da_path = extract_rda_object(TARBALL_PATH, "PROT_LIVER_DA")
prot_result = pyreadr.read_r(prot_da_path)
prot_df = prot_result[list(prot_result.keys())[0]]

# Same filtering logic as the transcriptomics version: 8-week timepoint, one row per gene,
# gene symbol mapping via the same FEATURE_TO_GENE_FILT table.
prot_timepoint_df = prot_df[prot_df["comparison_group"] == TIMEPOINT].copy()
prot_timepoint_df = prot_timepoint_df.sort_values("p_value").drop_duplicates(subset="feature_ID", keep="first")
prot_timepoint_df = prot_timepoint_df.merge(
    feature_to_gene[["feature_ID", "gene_symbol"]].drop_duplicates(subset="feature_ID"),
    on="feature_ID", how="left"
).dropna(subset=["gene_symbol"])

# Same "responsive" definition as transcriptomics: significant in either direction.
PROT_LFC_COL = "logFC" if "logFC" in prot_timepoint_df.columns else "shrunk_logFC"
prot_responsive_genes = set(
    prot_timepoint_df[prot_timepoint_df["p_value"] < SIG_THRESHOLD]["gene_symbol"]
)

# Compare against our transcriptomics-based responsive gene list from earlier in Part 2.
transcript_responsive_genes = set(responsive[GENE_COL])

both_responsive = transcript_responsive_genes & prot_responsive_genes
only_transcript = transcript_responsive_genes - prot_responsive_genes
only_protein = prot_responsive_genes - transcript_responsive_genes

print(f"Genes responsive by transcriptomics: {len(transcript_responsive_genes)}")
print(f"Genes responsive by proteomics: {len(prot_responsive_genes)}")
print(f"Responsive by BOTH: {len(both_responsive)}")
print(f"Responsive by transcriptomics ONLY: {len(only_transcript)}")
print(f"Responsive by proteomics ONLY: {len(only_protein)}")
print(f"\nExample genes responsive by both: {list(both_responsive)[:10]}")

Genes responsive by transcriptomics: 2593
Genes responsive by proteomics: 2917
Responsive by BOTH: 679
Responsive by transcriptomics ONLY: 1914
Responsive by proteomics ONLY: 2238

Example genes responsive by both: ['Dnaja4', 'Cryab', 'Fcgrt', 'Psmd14', 'Atxn10', 'Nsfl1c', 'Ube2g2', 'Pag1', 'Gzmb', 'Chmp3']


In [ ]:
print(f"Total genes measured (any p-value), transcriptomics: {timepoint_df['gene_symbol'].nunique() if 'gene_symbol' in timepoint_df.columns else 'need merge first'}")
print(f"Total genes measured (any p-value), proteomics: {prot_timepoint_df['gene_symbol'].nunique()}")

Total genes measured (any p-value), transcriptomics: 14015
Total genes measured (any p-value), proteomics: 9363


### Addendum results: transcript-level and protein-level responsiveness calls agree only about a quarter of the time

- **14,015** genes had transcriptomics measurements; **2,593** (19%) were called responsive
- **9,363** genes had proteomics measurements; **2,917** (31%) were called responsive — a higher
  hit *rate*, despite proteomics covering ~33% fewer genes overall (consistent with the
  narrower proteomics coverage noted in Part 2, but a higher fraction of what it does measure
  turning out significant)
- Only **679 genes** were called responsive by both layers — just **26%** of transcript-responsive
  genes and **23%** of protein-responsive genes overlap with the other layer's call

**This is a substantial, not minor, disagreement.** Note that this is an *overlap* between two
independent binary "responsive/not" calls, not a correlation between the two layers' continuous
effect sizes — we haven't tested whether genes that change *more* at the RNA level also change
more at the protein level, only whether each layer's own significance threshold agrees on which
genes count as responsive at all. Which molecular layer you use to define "exercise-responsive"
meaningfully changes which genes get that label. This is a real, important limitation for
interpreting Part 3 and Part 4: those sections model **protein sequence** with ESM2, but the
"responsive" label they're trying to predict was defined using **RNA-level** data. A gene ESM2
successfully predicts as "responsive" is being validated against a transcript-level definition
that may not reflect what's actually happening at the protein level for that same gene.